In [0]:
/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_1.csv
/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_HASH_1.csv
/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_Raw_data.csv

In [0]:
df_t_od_cs_hospital_raw_data = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_Raw_data.csv")

In [0]:
df_t_od_cs_hospital_raw_data = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_Raw_data.csv")

In [0]:
df_t_od_cs_hospitall_1 = spark.read.format("csv").option("delimiter", "|").option("header", "true").option("inferSchema", "true").load("/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_1.csv")

In [0]:
df_t_od_cs_hospitall_1 = (
    spark.read.format("csv")
    .option("delimiter", "|")
    .option("header", "true")
    .option("inferSchema", "false")
    # .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .load("/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_1.csv")
)

In [0]:
display(df_t_od_cs_hospitall_1)

In [0]:
from pyspark.sql import functions as F

path = "/Volumes/iics/casemix/history/hash_test/T_OD_CS_HOSPITAL_1.csv"

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "|")
    .option("quote", "")
    .option("inferSchema", "false")
    .load(path)
)

# Strip quotes from column names and from all values
df = df.toDF(*[c.replace('"', '') for c in df.columns])
df = df.select([F.regexp_replace(F.col(c), '"', '').alias(c) for c in df.columns])

display(df)

In [0]:
df = (
    df.withColumn("START_YEAR", F.col("START_YEAR").cast("int"))
      .withColumn("END_YEAR", F.when(F.col("END_YEAR") == "", None).otherwise(F.col("END_YEAR")).cast("int"))
      .withColumn("DH_CREATED_DATE", F.to_timestamp("DH_CREATED_DATE"))
      .withColumn("DH_UPDATED_DATE", F.to_timestamp("DH_UPDATED_DATE"))
)

df.printSchema()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("iics.casemix.t_od_cs_hospitall_1_test")

In [0]:
%sql
select * from iics.casemix.t_od_cs_hospitall_1_test

In [0]:
%sql
SELECT * FROM moh_raz.raz_casemix_raw.hashed_values